In [1]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def run_shap_workbench(trained_model, X_sample, customer_idx=0, top_n=20):
    """
    Hàm tất-cả-trong-một:
    - Vẽ 2 phương án biểu đồ Global SHAP thay thế Beeswarm.
    - Phân tích chi tiết hồ sơ cá nhân của khách hàng tại dòng `customer_idx`.
    """
    print("⏳ Đang khởi tạo TreeExplainer và tính toán SHAP values...")
    explainer = shap.TreeExplainer(trained_model)
    shap_values = explainer.shap_values(X_sample)

    # Đối với bài toán nhị phân, LightGBM thường trả về list [shap_class_0, shap_class_1] hoặc mảng 2D
    if isinstance(shap_values, list):
        shap_vals = shap_values[1]
    else:
        shap_vals = shap_values

    feature_names = list(X_sample.columns)

    # =========================================================================
    # PHẦN 1: SO SÁNH 2 BIỂU ĐỒ GLOBAL (THAY THẾ BEESWARM)
    # =========================================================================
    print(f"\n📊 Đang vẽ 2 phương án biểu đồ Global Importance cho Top {top_n} biến...")
    fig, axes = plt.subplots(1, 2, figsize=(20, 9))

    # --- PHƯƠNG ÁN 1: Mean Absolute Bar Plot ---
    mean_abs = np.mean(np.abs(shap_vals), axis=0)
    top_indices_1 = np.argsort(mean_abs)[-top_n:]

    y_pos1 = np.arange(top_n)
    axes[0].barh(y_pos1, mean_abs[top_indices_1], color='#1f77b4', edgecolor='black', alpha=0.85)
    axes[0].set_yticks(y_pos1)
    axes[0].set_yticklabels([feature_names[i] for i in top_indices_1], fontsize=10)
    axes[0].set_xlabel('Mean |SHAP Value| (Mức độ tác động tuyệt đối)', fontsize=11, fontweight='bold')
    axes[0].set_title('PHƯƠNG ÁN 1: Mean Absolute Bar Plot\n(Gọn gàng, chuẩn mực báo cáo kỹ thuật)', fontsize=12, fontweight='bold')
    axes[0].grid(axis='x', linestyle=':', alpha=0.6)

    # --- PHƯƠNG ÁN 2: Positive vs. Negative Impact Bar Plot ---
    pos_impact = np.mean(np.maximum(shap_vals, 0), axis=0)
    neg_impact = np.mean(np.minimum(shap_vals, 0), axis=0)
    total_impact = pos_impact + np.abs(neg_impact)
    top_indices_2 = np.argsort(total_impact)[-top_n:]

    y_pos2 = np.arange(top_n)
    axes[1].barh(y_pos2, pos_impact[top_indices_2], color='#d62728', alpha=0.85, edgecolor='black', label='Đẩy rủi ro tăng (+PD)')
    axes[1].barh(y_pos2, neg_impact[top_indices_2], color='#2ca02c', alpha=0.85, edgecolor='black', label='Kéo rủi ro giảm (-PD)')
    axes[1].axvline(0, color='black', linewidth=1)
    axes[1].set_yticks(y_pos2)
    axes[1].set_yticklabels([feature_names[i] for i in top_indices_2], fontsize=10)
    axes[1].set_xlabel('Mean Impact on Model Output (Chiều tác động)', fontsize=11, fontweight='bold')
    axes[1].set_title('PHƯƠNG ÁN 2: Positive vs. Negative Decomposition\n(Thấy rõ biến nào chuyên làm tăng/giảm rủi ro)', fontsize=12, fontweight='bold')
    axes[1].legend(loc='lower right', fontsize=10)
    axes[1].grid(axis='x', linestyle=':', alpha=0.6)

    plt.tight_layout()
    plt.show()

    # =========================================================================
    # PHẦN 2: BÓC TÁCH HỒ SƠ CÁ NHÂN (LOCAL EXPLANATION)
    # =========================================================================
    row_shap = shap_vals[customer_idx]
    row_data = X_sample.iloc[customer_idx]

    df_local = pd.DataFrame({
        'Feature': feature_names,
        'Value': row_data.values,
        'SHAP_Impact': row_shap
    })

    # Tách top 5 lý do làm tăng rủi ro và top 5 lý do kéo giảm rủi ro
    df_risk_up = df_local[df_local['SHAP_Impact'] > 0].sort_values(by='SHAP_Impact', ascending=False).head(5)
    df_risk_down = df_local[df_local['SHAP_Impact'] < 0].sort_values(by='SHAP_Impact', ascending=True).head(5)

    print("=" * 80)
    print(f"📋 PHÂN TÍCH HỒ SƠ KHÁCH HÀNG (Dòng index: {customer_idx})")
    print("=" * 80)

    print("\n🔴 [TOP 5 YẾU TỐ LÀM TĂNG RỦI RƠ - REASON CODES TỪ CHỐI]:")
    print(df_risk_up.to_string(index=False))

    print("\n🟢 [TOP 5 ĐIỂM TÍCH CỰC - YẾU TỐ BẢO VỆ GIẢM RỦI RO]:")
    print(df_risk_down.to_string(index=False))
    print("=" * 80)

    return {
        'explainer': explainer,
        'shap_values': shap_vals,
        'customer_risk_up': df_risk_up,
        'customer_risk_down': df_risk_down
    }

/Users/nguyenminhtri/FinalYearPro/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Lấy mô hình LightGBM fold đầu tiên đã train
model_to_explain = results['lgb_models'][0]

# 2. Chuẩn bị tập dữ liệu mẫu (chỉ cần lấy 1000 dòng để tính SHAP cho nhanh, không bị đơ RAM)
df_eval = pd.read_parquet('/data/processed/df_train_fair_compliant.parquet')
X_eval = df_eval.drop(columns=[col for col in ['SK_ID_CURR', 'TARGET'] if col in df_eval.columns])

# Đảm bảo đúng định dạng category
cat_cols = X_eval.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    X_eval[col] = X_eval[col].astype('category')

sample_X = X_eval.sample(n=1000, random_state=42)

# 3. Chạy toàn bộ phân tích SHAP
shap_results = run_shap_workbench(
    trained_model=model_to_explain,
    X_sample=sample_X,
    customer_idx=0, # Xem khách hàng đầu tiên trong mẫu
    top_n=20
)

NameError: name 'results' is not defined